# 47 - Open VLA Models And Action Heads

## What / Why / How

**What we are trying to do:** Compare open VLA policy families such as OpenVLA, SmolVLA, pi0/pi0.5, GR00T-style models, Xiaomi-Robotics-0, and Dexora.

**Why this matters:** Modern robotics is increasingly about action representation: regression, action chunks, flow matching, autoregression, discrete diffusion, and real-time chunking.

**How we will do it:** Use toy action-head simulations to understand latency, multimodality, and action smoothing before touching large models.

## Open VLA Model Families

Current open VLA systems differ most in action representation and deployment strategy:

- OpenVLA: large open VLA with fine-tuning path.
- SmolVLA: compact, accessible VLA.
- OpenPI pi0/pi0.5: flow matching and pi0-FAST autoregressive variants.
- GR00T-style models: humanoid-oriented foundation policies integrated with LeRobot.
- Xiaomi-Robotics-0: real-time asynchronous deployment emphasis.
- Dexora: high-DoF bimanual and dexterous manipulation.

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
action_heads = {
    "mean_regression": {"multimodal": 0.2, "latency": 0.95, "smoothness": 0.65, "difficulty": 0.2},
    "action_chunking": {"multimodal": 0.45, "latency": 0.85, "smoothness": 0.85, "difficulty": 0.35},
    "diffusion": {"multimodal": 0.9, "latency": 0.35, "smoothness": 0.8, "difficulty": 0.75},
    "flow_matching": {"multimodal": 0.85, "latency": 0.55, "smoothness": 0.85, "difficulty": 0.75},
    "autoregressive_tokens": {"multimodal": 0.7, "latency": 0.6, "smoothness": 0.65, "difficulty": 0.65},
    "real_time_chunking": {"multimodal": 0.65, "latency": 0.85, "smoothness": 0.9, "difficulty": 0.65},
}
for name, attrs in action_heads.items():
    print(name, attrs)

In [ ]:
# Toy latency budget: a slow VLA predicts chunks, a fast controller consumes them.
control_hz = 50
chunk_horizon = 10
inference_ms = {
    "SmolVLA-ish": 45,
    "pi0.5-ish": 120,
    "large VLA-ish": 260,
}
for model, ms in inference_ms.items():
    chunk_duration_ms = 1000 * chunk_horizon / control_hz
    margin = chunk_duration_ms - ms
    print(model, "chunk_duration_ms", chunk_duration_ms, "inference_margin_ms", margin)

In [ ]:
# Multimodal action example: go above or below an obstacle.
rng = np.random.default_rng(47)
modes = rng.choice([-1, 1], size=400)
actions = np.c_[rng.normal(0.8, 0.04, 400), modes * rng.normal(0.55, 0.04, 400)]
mean_action = actions.mean(axis=0)
sampled = actions[rng.choice(len(actions), 6, replace=False)]
print("mean action:", mean_action)
print("mean crosses obstacle band?", abs(mean_action[1]) < 0.2)
print("sampled safe actions:")
print(sampled)

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(5, 4))
    plt.scatter(actions[:, 0], actions[:, 1], s=8, alpha=0.25)
    plt.scatter([mean_action[0]], [mean_action[1]], c="tab:red", s=80, label="mean")
    plt.axhspan(-0.2, 0.2, color="black", alpha=0.1, label="unsafe band")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    plot_unavailable()

## Exercises

1. Add a discrete diffusion action head to the table.
2. Change control frequency to 200 Hz and inspect latency pressure.
3. Write which VLA family you would try first on a small arm and why.